# Concept Drift Detection — DDM, EDDM & Page-Hinkley Test

Concept drift occurs when the **statistical properties of the target concept change over time**, causing the trained model to become stale.

---

## Drift Taxonomy

| Type | Description | Example |
|---|---|---|
| **Abrupt** | Sudden shift in distribution | Sensor failure |
| **Gradual** | Old concept fades, new one strengthens | Consumer behaviour shift |
| **Incremental** | Smooth continuous change | Climate change |
| **Recurring** | Old concept reappears | Seasonal patterns |

---

## 1  DDM — Drift Detection Method (Gama et al., 2004)

Monitors the **online error rate** $p_i$ (Bernoulli proportion).  
By the Central Limit Theorem, the standard deviation of $p_i$ is $s_i = \sqrt{p_i(1-p_i)/i}$.

Track $p_{min}$ and $s_{min}$ (lowest error seen so far):

| Condition | Action |
|---|---|
| $p_i + s_i \geq p_{min} + 2 \cdot s_{min}$ | **Warning** zone |
| $p_i + s_i \geq p_{min} + 3 \cdot s_{min}$ | **Drift** detected |

---

## 2  EDDM — Early Drift Detection Method (Baena-García et al., 2006)

Instead of monitoring the error rate, EDDM monitors the **distance between two consecutive errors** $d_i$.  
More errors far apart → stable; errors close together → drift.

Tracks mean $\mu_d$ and std $\sigma_d$ of inter-error distances:

| Condition | Action |
|---|---|
| $(\mu_d + 2\sigma_d) / (\mu_{d,max} + 2\sigma_{d,max}) < \alpha$ | **Warning** |
| $(\mu_d + 2\sigma_d) / (\mu_{d,max} + 2\sigma_{d,max}) < \beta$ | **Drift** |

---

## 3  Page-Hinkley Test (PHT)

A sequential change-point test based on cumulative sum (CUSUM).  
Detects a **rise** in the cumulative mean deviation:

$$m_T = \sum_{t=1}^T (x_t - \bar{x}_T - \delta)$$

$$M_T = \max_{1 \leq i \leq T} m_i$$

Drift when $M_T - m_T > \lambda$ (threshold).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
print('Imports OK')

---
## Implementations

In [ ]:
class DDM:
    """Drift Detection Method (Gama et al., 2004)."""

    WARNING_LEVEL = 2.0
    DRIFT_LEVEL   = 3.0
    MIN_INSTANCES = 30

    def __init__(self):
        self._reset()

    def _reset(self):
        self.n = 0
        self.p = 1.0
        self.s = 0.0
        self.p_min = np.inf
        self.s_min = np.inf
        self.in_warning = False

    def add_element(self, error: int) -> str:
        """error: 1 if misclassified, 0 if correct."""
        self.n += 1
        self.p  = self.p + (error - self.p) / self.n
        self.s  = np.sqrt(self.p * (1 - self.p) / self.n)

        if self.n < self.MIN_INSTANCES:
            return 'stable'

        if self.p + self.s < self.p_min + self.s_min:
            self.p_min = self.p
            self.s_min = self.s

        stat = self.p + self.s
        base = self.p_min + self.s_min

        if stat >= base + self.DRIFT_LEVEL * self.s_min:
            self._reset()
            return 'drift'
        elif stat >= base + self.WARNING_LEVEL * self.s_min:
            self.in_warning = True
            return 'warning'
        else:
            self.in_warning = False
            return 'stable'


class EDDM:
    """Early Drift Detection Method (Baena-García et al., 2006)."""

    ALPHA = 0.95
    BETA  = 0.90
    MIN_ERRORS = 30

    def __init__(self):
        self._reset()

    def _reset(self):
        self.n = 0
        self.n_errors = 0
        self.last_error_pos = 0
        self.distances: list[int] = []
        self.mu_max = 0.0
        self.sigma_max = 0.0

    def add_element(self, error: int) -> str:
        self.n += 1
        if not error:
            return 'stable'

        self.n_errors += 1
        d = self.n - self.last_error_pos
        self.last_error_pos = self.n
        self.distances.append(d)

        if self.n_errors < self.MIN_ERRORS:
            return 'stable'

        mu = np.mean(self.distances)
        sigma = np.std(self.distances)
        metric = mu + 2 * sigma

        if metric > self.mu_max + 2 * self.sigma_max:
            self.mu_max    = mu
            self.sigma_max = sigma

        ratio = metric / (self.mu_max + 2 * self.sigma_max + 1e-9)

        if ratio < self.BETA:
            self._reset()
            return 'drift'
        elif ratio < self.ALPHA:
            return 'warning'
        return 'stable'


class PageHinkley:
    """Page-Hinkley Test for mean change detection."""

    def __init__(self, delta: float = 0.005, lam: float = 50.0, alpha: float = 1 - 0.0001):
        self.delta = delta   # allowed error in mean estimate
        self.lam   = lam     # detection threshold
        self.alpha = alpha   # forgetting factor
        self._reset()

    def _reset(self):
        self.n = 0
        self.sum = 0.0
        self.x_mean = 0.0
        self.m_t = 0.0
        self.M_t = 0.0

    def add_element(self, x: float) -> str:
        self.n += 1
        self.x_mean = self.alpha * self.x_mean + (1 - self.alpha) * x
        self.m_t += x - self.x_mean - self.delta
        self.M_t  = max(self.M_t, self.m_t)

        if self.M_t - self.m_t > self.lam:
            self._reset()
            return 'drift'
        return 'stable'


print('DDM, EDDM, PageHinkley ready.')

---
## Comparison on Synthetic Streams

In [ ]:
def simulate_error_stream(n=2000, drift_at=1000, p_before=0.1, p_after=0.4, seed=42):
    """Binary error stream: error rate jumps at drift_at."""
    rng = np.random.default_rng(seed)
    return np.concatenate([
        rng.binomial(1, p_before, drift_at),
        rng.binomial(1, p_after,  n - drift_at),
    ])


errors = simulate_error_stream()
TRUE_DRIFT = 1000

detectors = {'DDM': DDM(), 'EDDM': EDDM()}
results = {name: {'drift': [], 'warning': []} for name in detectors}

for t, err in enumerate(errors):
    for name, det in detectors.items():
        status = det.add_element(int(err))
        if status == 'drift':
            results[name]['drift'].append(t)
        elif status == 'warning':
            results[name]['warning'].append(t)

for name, res in results.items():
    first_post = [d for d in res['drift'] if d >= TRUE_DRIFT]
    lag = first_post[0] - TRUE_DRIFT if first_post else 'missed'
    fp = len([d for d in res['drift'] if d < TRUE_DRIFT])
    print(f'{name}: first post-drift detection at {first_post[0] if first_post else "—"} '
          f'(lag={lag}), false alarms={fp}')

In [ ]:
# Visualise DDM and EDDM detections
rolling_err = pd.Series(errors.astype(float)).rolling(50).mean()

fig, axes = plt.subplots(2, 1, figsize=(13, 7), sharex=True)

for ax, name in zip(axes, ['DDM', 'EDDM']):
    ax.plot(rolling_err, alpha=0.7, lw=1.5, label='Rolling error rate (w=50)')
    for d in results[name]['warning'][:40]:
        ax.axvline(d, color='orange', alpha=0.3, lw=0.6)
    for d in results[name]['drift']:
        ax.axvline(d, color='red', alpha=0.5, lw=1.0)
    ax.axvline(TRUE_DRIFT, color='black', ls='--', lw=2, label='True drift')
    ax.set_title(f'{name} — detections (red=drift, orange=warning)')
    ax.legend()

plt.tight_layout()
plt.show()

---
## Page-Hinkley on Value Stream

In [ ]:
# PHT works on raw values, not error rates
value_stream = np.concatenate([
    np.random.normal(0, 1, 500),
    np.random.normal(2, 1, 500),
    np.random.normal(-1, 1, 500),
])
true_drifts_pht = [500, 1000]

pht = PageHinkley(delta=0.005, lam=30)
pht_drifts = []

for t, val in enumerate(value_stream):
    if pht.add_element(val) == 'drift':
        pht_drifts.append(t)

print(f'PHT detections: {pht_drifts}  (true: {true_drifts_pht})')

plt.figure(figsize=(12, 4))
plt.plot(value_stream, alpha=0.5, lw=0.8)
for td in true_drifts_pht:
    plt.axvline(td, color='black', ls='--', lw=2)
for d in pht_drifts:
    plt.axvline(d, color='red', lw=1.5, alpha=0.7)
plt.title('Page-Hinkley Test — Drift Detection (red=detected, dashed=true)')
plt.tight_layout()
plt.show()

---
## Detector Comparison Summary

| Detector | Monitors | Best for | Key parameter |
|---|---|---|---|
| **DDM** | Error rate $p_i + s_i$ | Abrupt drifts in classifiers | Warning/Drift multipliers (2, 3) |
| **EDDM** | Inter-error distances | Gradual / early drifts | α=0.95, β=0.90 |
| **PHT** | Raw value CUSUM | Regression / value streams | δ (sensitivity), λ (threshold) |
| **ADWIN** | Any statistic, adaptive window | General-purpose | δ (confidence) |

**See also:** Topic 119 (ADWIN), Topic 120 (VFDT/CVFDT), Topic 117 (stream adaptive learning)